# Step 1. 강원도 기후지형 및 캐나다 지수 기본분석

산불 Target을 사용하지 않고 강원도 기상셀의 평상시 기후, 지형, 캐나다 지수 배경을 준비한다.

## 현재 구현 범위

이 노트북은 **원천 데이터 경로 확인, 로딩, 필수 스키마 검증, 시간 파싱,
기본 행 수 감사**까지 구현한다. 통계 분석과 시각화는 이후 셀에서 이어서 작성한다.

공통 데이터 계약은 `README.md`를 따른다. Step 2~6에서 캐나다 지수를 사건 시각에
결합할 때는 12시 이전이면 전일 정오, 12시 이후이면 당일 정오 자료만 사용한다.

결과 해석은 이 노트북에 작성하지 않는다. 결과표와 플롯을 함께 검토한 해석,
한계와 다음 코드 반영사항은 대응 `진행예정로그.md`에만 기록한다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
candidates = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
REPO_ROOT = next(
    (path for path in candidates if (path / "jsw/강원_재_EDA/re_eda_common.py").exists()),
    Path(r"D:/farm-system-public-02"),
)
MODULE_DIR = REPO_ROOT / "jsw/강원_재_EDA"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from re_eda_common import (
    DATA_PATHS,
    DERIVED_WEATHER_COLUMNS,
    RAW_WEATHER_COLUMNS,
    add_canadian_asof_keys,
    check_sources,
    configure_notebook,
    frame_inventory,
    load_access_lines,
    load_canadian_indices,
    load_dem_metadata,
    load_fire,
    load_grid_bundle,
    load_hourly_weather,
    load_infrastructure,
    load_landcover,
    load_roads,
    load_terrain,
)

configure_notebook()

## 1. 원천 파일 존재 여부

In [ ]:
SOURCE_KEYS = ['weather_cells', 'weather_grid', 'weather_hourly_raw', 'climate_type', 'canada_ffmc', 'canada_fwi', 'dem']
source_audit = check_sources(SOURCE_KEYS)
display(source_audit)

## 2. 기상셀, 기후지형유형, 공간격자 로딩

In [ ]:
grid_bundle = load_grid_bundle()
weather_cells = grid_bundle["cells"]
climate_type = grid_bundle["climate"]
weather_grid = grid_bundle["grid"]

display(weather_cells.head())
display(climate_type["기후지형유형"].value_counts(dropna=False))
print("기상 격자 CRS:", weather_grid.crs)

## 3. 원본 시간단위 기상 로딩

In [ ]:
hourly_weather = load_hourly_weather(
    derived=False,
    columns=RAW_WEATHER_COLUMNS,
)
print("기간:", hourly_weather["일시"].min(), "~", hourly_weather["일시"].max())
print("기상셀 수:", hourly_weather["기상셀ID"].nunique())
display(hourly_weather.head())

## 4. 캐나다 산불위험지수 로딩

In [ ]:
canadian_indices = load_canadian_indices()
print("기간:", canadian_indices["날짜"].min(), "~", canadian_indices["날짜"].max())
print("기상셀 수:", canadian_indices["기상셀ID"].nunique())
display(canadian_indices.head())

## 5. DEM 메타데이터 로딩

In [ ]:
dem_metadata = load_dem_metadata()
display(dem_metadata)

## 로딩 결과 요약

In [ ]:
loaded_frames = {
    "weather_cells": weather_cells,
    "climate_type": climate_type,
    "weather_grid": weather_grid,
    "hourly_weather": hourly_weather,
    "canadian_indices": canadian_indices,
}
display(frame_inventory(loaded_frames))

## 다음 구현 범위

기상셀 중심점의 DEM 고도 샘플링, 월·계절 파생, 영동/영서 검정과 K=2·3 군집 탐색을 구현한다.

현재 노트북은 로딩과 입력 감사까지만 실행한다. 이후 분석 셀에서도
`README.md`의 미래 정보 누수 방지 규칙과 대조군 정의를 유지해야 한다.
실행 결과의 해석은 대응 `진행예정로그.md`에 작성한다.